In [1]:
import random
import torch
import os
import umap
import time
import glob
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.decomposition import PCA, FastICA, FactorAnalysis
from sklearn.random_projection import GaussianRandomProjection
from sklearn.manifold import TSNE, trustworthiness
from sklearn.neighbors import NearestNeighbors


from sklearn.model_selection import (
    StratifiedKFold, cross_validate, GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, make_scorer, confusion_matrix
)

from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB


In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
# Helper function to make embeddings from chosen dimensionality reduction algorithm.
def put_embeddings_in_df(input_df, method):
    model = method(n_components=2, random_state=RANDOM_STATE)

    trans_embeddings = model.fit_transform(input_df[[f'factor_{i}' for i in range(1, 7)]].values)

    input_df['x'] = [trans_embedding[0] for trans_embedding in trans_embeddings]
    input_df['y'] = [trans_embedding[1] for trans_embedding in trans_embeddings]
    return input_df


In [5]:
# Get all CSV files.
all_files = glob.glob("./outputsTrain/*/mean_model_zero_shot_classification.csv")  # change this to your folder path

# Loop through files.
all_dfs = []

for f in all_files:
    df = pd.read_csv(f)
    # Extract categories from doc_id
    doc_ids = df['doc_id']
    df['category'] = df['doc_id'].apply(lambda x: x.split('_')[0])
    
    # Aggregate numeric factor columns by category
    factor_cols = [f"factor_{i}" for i in range(1, 7)]
    df_agg = df.groupby('category')[factor_cols].mean().reset_index()
    df_agg['doc_id'] = doc_ids
    all_dfs.append(df_agg)

# Concatenate all aggregated DataFrames
dfs = pd.concat(all_dfs, ignore_index=True)
# Use UMAP as chosen method.
dfs = put_embeddings_in_df(dfs, umap.UMAP)

/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
# get models - broad coverage across linear, margin, instance-based, and nonlinear tree ensembles
def build_models(random_state):
    models = {
        # high numbers of iterations (max_iter) avoids convergence warnings in high-dimensional spaces
        "LogisticRegression": LogisticRegression(max_iter=20000, class_weight="balanced", random_state=random_state), # strong baseline
        "RidgeClassifier": RidgeClassifier(class_weight="balanced", random_state=random_state), # strong baseline
        "LinearSVC": LinearSVC(max_iter=20000, class_weight="balanced", random_state=random_state), # strong baseline
        "SVC-RBF": SVC(probability=False, random_state=random_state, class_weight="balanced"), # probability set to false saves on compute
        "SGD-Hinge": SGDClassifier(loss="hinge", max_iter=20000, random_state=random_state, class_weight="balanced"), # hinge loss is good for binary classification problems - fast on large data
        "KNN": KNeighborsClassifier(), # deterministic - sensitive to scaling (good to test with and without standard scaler)
        "DecisionTree": DecisionTreeClassifier(random_state=random_state, class_weight="balanced"), # don't need scaling and captures nonlinear splits
        "RandomForest": RandomForestClassifier(random_state=random_state, class_weight="balanced"), # don't need scaling and captures nonlinear splits
        "ExtraTrees": ExtraTreesClassifier(random_state=random_state, class_weight="balanced"), # don't need scaling and captures nonlinear splits
        "GradientBoosting": GradientBoostingClassifier(random_state=random_state), # don't need scaling and captures nonlinear splits
        "HistGB": HistGradientBoostingClassifier(random_state=random_state), # don't need scaling and captures nonlinear splits
        "AdaBoost": AdaBoostClassifier(random_state=random_state), # don't need scaling and captures nonlinear splits
        "GaussianNaiveBayes": GaussianNB(), # deterministic - interpretable
    }
    return models

# get parameter grid to test
def small_param_grid(name): # high-leverage parameter sweeps (runtime will be effected if we test everything)
    grids = {
        # lowered c values due to convergence issues
        "LogisticRegression": {"clf__C": [0.1, 0.5, 1.0]},
        "LinearSVC":          {"clf__C": [0.1, 0.5, 1.0]},
        "SVC-RBF":            {"clf__C": [0.1, 0.5, 1.0], "clf__gamma": ["scale", "auto"]},
        "KNN":                {"clf__n_neighbors": [3, 5, 11]},
        "RandomForest":       {"clf__n_estimators": [300, 600], "clf__max_depth": [None, 10, 20]},
        "ExtraTrees":         {"clf__n_estimators": [300, 600], "clf__max_depth": [None, 10, 20]},
        "HistGB":             {"clf__max_depth": [None, 6, 10]},
    }
    return grids.get(name, None)

# make pipeline with and without scaling
def preprocessor(scale):
    # median ensures it is robust to outliers
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        # scales data to benefit models such as SVM and KNN
        steps.append(("scaler", StandardScaler(with_mean=True)))
    else:
        steps.append(("scaler", FunctionTransformer(lambda X: X, feature_names_out="one-to-one")))
    return Pipeline(steps)


def gmean_binary_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0   # sensitivity
    tnr = tn / (tn + fp) if (tn + fp) else 0.0   # specificity
    return np.sqrt(tpr * tnr)

gmean_scorer = make_scorer(gmean_binary_score)

def build_scorers():
    return {
        "accuracy": "accuracy", # normal accuracy
        "balanced_accuracy": make_scorer(balanced_accuracy_score), # balanced accuracy
        "precision": make_scorer(precision_score, average="binary", zero_division=0),
        "recall": make_scorer(recall_score, average="binary", zero_division=0),
        "f1": make_scorer(f1_score, average="binary", zero_division=0),
        "roc_auc": "roc_auc",
        "gmean": gmean_scorer
    }

In [ ]:
def evaluate_one(X_train, y_train, X_test, y_test, feature_name, variant,
                 random_state=RANDOM_STATE, n_splits=N_SPLITS_CV, n_jobs=N_JOBS):

    inner_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    outer_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scorers = build_scorers()
    results = []

    for name, clf in build_models(random_state).items():
        # scaling decision
        X_tr, X_te = X_train, X_test

        # Convert sparse to dense for models that need dense arrays
        if sparse.issparse(X_tr):
            if name in ["HistGB", "GradientBoosting", "GaussianNaiveBayes"]:
                X_tr = X_tr.toarray()
                X_te = X_te.toarray()

        # Scaling decision
        if variant == "scaled":
            scaler = StandardScaler(with_mean=False if sparse.issparse(X_tr) else True)
        else:
            scaler = FunctionTransformer(lambda X: X, feature_names_out="one-to-one")

        pipe = Pipeline([
            ("scale", scaler),
            ("clf", clf)
        ])

        grid = small_param_grid(name)
        model_for_cv = GridSearchCV(pipe, grid, scoring=scorers, cv=inner_cv,
                                    n_jobs=n_jobs, refit="gmean") if grid else pipe

        cv_out = cross_validate(model_for_cv, X_tr, y_train, cv=outer_cv,
                                scoring=scorers, return_train_score=False,
                                n_jobs=n_jobs, return_estimator=True)

        # Fit on full training
        model_for_cv.fit(X_tr, y_train)
        y_pred = model_for_cv.predict(X_te)

        # best estimator info
        if isinstance(model_for_cv, GridSearchCV):
            chosen = model_for_cv.best_estimator_
            tuned = model_for_cv.best_params_
        else:
            chosen = model_for_cv
            tuned = {}

        clf_params = chosen.named_steps["clf"].get_params()
        tuned_json = json.dumps(tuned, default=str)
        clf_json = json.dumps(clf_params, default=str)

        y_score = None
        if hasattr(model_for_cv, "predict_proba"):
            y_score = model_for_cv.predict_proba(X_te)[:, 1]
        elif hasattr(model_for_cv, "decision_function"):
            y_score = model_for_cv.decision_function(X_te)

        row = {
            "feature_set": feature_name,
            "variant": variant,
            "model": name,
            "tuned_parameters": tuned_json,
            "clf_parameters": clf_json,
            **{f"cv_{k.replace('test_','')}_mean": float(np.mean(v))
               for k,v in cv_out.items() if k.startswith("test_")},
            **{f"cv_{k.replace('test_','')}_std": float(np.std(v))
               for k,v in cv_out.items() if k.startswith("test_")},
            "holdout_accuracy": accuracy_score(y_test, y_pred),
            "holdout_bal_acc": balanced_accuracy_score(y_test, y_pred),
            "holdout_precision": precision_score(y_test, y_pred, zero_division=0),
            "holdout_recall": recall_score(y_test, y_pred, zero_division=0),
            "holdout_f1": f1_score(y_test, y_pred, zero_division=0),
            "holdout_roc_auc": roc_auc_score(y_test, y_score) if y_score is not None else np.nan,
            "holdout_gmean": gmean_binary_score(y_test, y_pred)
        }

        results.append(row)
        print(row)
    return pd.DataFrame(results)

In [ ]:
all_results = []
for feature_name, X_train in inputs_dict.items():
    X_test = test_dict[feature_name]

    for variant in ("raw", "scaled"):
        df_res = evaluate_one(X_train, y_train, X_test, y_test, feature_name, variant)
        all_results.append(df_res)

# Concatenate all results and save.
final_results = pd.concat(all_results, ignore_index=True)
final_results.to_csv("baseline_model_results.csv", index=False)

In [ ]:
print(f"Highest: {final_results.sort_values(by='holdout_accuracy', ascending=False)['holdout_accuracy'].tolist()[0]}")
final_results.sort_values(by='holdout_accuracy', ascending=False).head()

In [ ]:
print(f"Highest: {final_results.sort_values(by='holdout_bal_acc', ascending=False)['holdout_bal_acc'].tolist()[0]}")
final_results.sort_values(by='holdout_bal_acc', ascending=False).head()

In [ ]:
print(f"Highest: {final_results.sort_values(by='holdout_precision', ascending=False)['holdout_precision'].tolist()[0]}")
final_results.sort_values(by='holdout_precision', ascending=False).head()

In [ ]:
print(f"Highest: {final_results.sort_values(by='holdout_recall', ascending=False)['holdout_recall'].tolist()[0]}")
final_results.sort_values(by='holdout_recall', ascending=False).head()

In [ ]:
print(f"Highest: {final_results.sort_values(by='holdout_f1', ascending=False)['holdout_f1'].tolist()[0]}")
final_results.sort_values(by='holdout_f1', ascending=False).head()

In [ ]:
print(f"Highest: {final_results.sort_values(by='holdout_roc_auc', ascending=False)['holdout_roc_auc'].tolist()[0]}")
final_results.sort_values(by='holdout_roc_auc', ascending=False).head()

In [ ]:
print(f"Highest: {final_results.sort_values(by='holdout_gmean', ascending=False)['holdout_gmean'].tolist()[0]}")
final_results.sort_values(by='holdout_gmean', ascending=False).head()